# Burunov Bot — F5-TTS на ЛУЧШИХ ручных транскрипциях

**Что изменилось:** кореш вручную перетранскрибировал топ-30 сегментов. Из них выбраны лучшие по рейтингу + чистоте звука (без женского голоса, без фоновой музыки, без обрезок).

**Главный референс:** `0043.wav` (rating 5/5) — «С одной стороны, я никогда не мечтал...»

Также прогоним 4 альтернативных референса и сравним какой даёт лучший клон Бурунова.

**GPU:** Colab T4 (бесплатно)

## 1. Установка F5-TTS

In [ ]:
!git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS
%cd /content/F5-TTS
!pip install -e . 2>&1 | tail -5
!pip install huggingface_hub soundfile torchaudio

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ Включи T4 GPU: Runtime → Change runtime type')

## 2. Загрузка датасета burunov-voice (для wav-файлов)

Нужны только 5 конкретных wav-файлов из `burunov-only-v2/segments/`. Можно скачать весь репо (медленно) или только нужные файлы (быстро).

In [ ]:
import os, json, urllib.request

BEST_REFS = [
    {
        'id': 'ref_0043',
        'wav': '-211220744_456239510_0043.wav',
        'duration': 8.1,
        'rating': 5,
        'text': 'С одной стороны, я никогда не мечтал даже об этом, не грезил и не ставил никаких целей, что произойдет так.',
        'primary': True,
    },
    {
        'id': 'ref_0293',
        'wav': '-211220744_456239510_0293.wav',
        'duration': 9.8,
        'rating': 4,
        'text': 'Главное, чтобы они были здоровые. Я реагирую как зависимый человек. У меня психика устроена и алгоритмы моей психики работают как у зависимого человека. Да.',
    },
    {
        'id': 'ref_0054',
        'wav': '-211220744_456239510_0054.wav',
        'duration': 8.0,
        'rating': 4,
        'text': 'Перепрошить решил, да, все это, чтобы обсуждалось, но смысл, но я буду нелепый с этими волосами. Да и меня не знают как бы с волосами-то.',
    },
    {
        'id': 'ref_0025',
        'wav': '-211220744_456239510_0025.wav',
        'duration': 8.0,
        'rating': 4,
        'text': 'Металлики, потому что я люблю металлику. Последний раз я был в Лужниках, когда они были в Москве в 2019 году. И голову так поворачиваю, и мне человек кивает.',
    },
    {
        'id': 'ref_0292',
        'wav': '-211220744_456239510_0292.wav',
        'duration': 9.8,
        'rating': 4,
        'text': 'С потухшим глазом и в том состоянии, в котором и физически, и в моральном я находился. Счастливое это отношение или не счастливое? Это покажет время.',
    },
]

BASE_URL = 'https://raw.githubusercontent.com/shray77/burunov-voice/main/burunov-only-v2/segments'
REFS_DIR = '/content/refs'
os.makedirs(REFS_DIR, exist_ok=True)

print('Скачиваем 5 wav-файлов напрямую с GitHub...')
for ref in BEST_REFS:
    out_path = os.path.join(REFS_DIR, ref['wav'])
    if os.path.exists(out_path):
        print(f'  ✓ {ref["wav"]} (уже есть)')
        continue
    url = f'{BASE_URL}/{ref["wav"]}'
    print(f'  → {ref["wav"]} (rating={ref["rating"]})')
    urllib.request.urlretrieve(url, out_path)
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'    {size_mb:.1f} MB')

print('\n✅ Все wav скачаны в', REFS_DIR)

## 3. Послушать все 5 референсов (опционально)

Перед запуском синтеза — проверь что слышишь чистый Бурунов без шумов.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

for ref in BEST_REFS:
    path = os.path.join(REFS_DIR, ref['wav'])
    audio, sr = sf.read(path)
    print(f'\n>>> {ref["id"]} ({ref["duration"]}сек, rating={ref["rating"]}/5)')
    print(f'    {ref["text"]}')
    display(Audio(path))

## 4. Скачать модель + инициализация F5-TTS

In [ ]:
from huggingface_hub import snapshot_download
print('Скачиваем F5-TTS-Russian...')
model_path = snapshot_download(repo_id='hotstone228/F5-TTS-Russian')
print(f'✅ {model_path}')

In [ ]:
import sys
sys.path.insert(0, '/content/F5-TTS/src')

from f5_tts.api import F5TTS
print('Инициализация F5TTS...')
tts = F5TTS(device='cuda' if torch.cuda.is_available() else 'cpu')
print('✅ F5-TTS готова')

## 5. Тест синтеза с каждым референсом

Прогоняем 5 тестовых фраз через каждый из 5 референсов (25 синтезов).
После — посмотрим какой референс даёт лучший клон.

Тестовые фразы — из сценария Бурунова (анекдоты + кофе).

In [ ]:
import time
import soundfile as sf

TEST_TEXTS = [
    'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'Бля, тут кто-то стоит... дай пройти.',
    'Вот ваш кофе, Олег. Не обожгись, бля.',
    'Учительница спрашивает Вовочку: какую оценку тебе поставить? Пять, Марьиванна! Ладно, четыре, но не меньше!',
]

os.makedirs('/content/compare', exist_ok=True)
results = []

for ref in BEST_REFS:
    ref_path = os.path.join(REFS_DIR, ref['wav'])
    print(f'\n========== РЕФЕРЕНС: {ref["id"]} (rating={ref["rating"]}/5) ==========')
    print(f'    Текст референса: {ref["text"][:80]}...')
    
    for i, test_text in enumerate(TEST_TEXTS):
        out = f'/content/compare/{ref["id"]}_test{i}.wav'
        print(f'\n  → [{i+1}/5] "{test_text[:60]}..."')
        
        t0 = time.time()
        try:
            wav, sr, _ = tts.infer(
                ref_file=ref_path,
                ref_text=ref['text'],
                gen_text=test_text,
                file_wave=out,
            )
            dt = time.time() - t0
            audio, sr_out = sf.read(out)
            audio_dur = len(audio) / sr_out
            rtf = dt / audio_dur
            print(f'    ✅ Синтез {dt:.1f}с | Аудио {audio_dur:.1f}с | RTF={rtf:.2f}')
            
            results.append({
                'ref_id': ref['id'],
                'ref_rating': ref['rating'],
                'test_idx': i,
                'test_text': test_text,
                'synth_time': dt,
                'audio_dur': audio_dur,
                'rtf': rtf,
                'path': out,
            })
        except Exception as e:
            print(f'    ❌ Ошибка: {e}')
            import traceback; traceback.print_exc()

print(f'\n✅ Готово, синтезировано {len(results)} аудио')

## 6. Прослушать все результаты — выбрать лучший референс

Для каждого референса проигрываем одну и ту же фразу — сравниваем какой клон лучше.

Фраза: «Угу, щас, Олег Тарасыч... кофеварку найду.»

In [ ]:
# Сравнение на первой тестовой фразе
TEST_IDX = 0
print(f'\n>>> Тестовая фраза: "{TEST_TEXTS[TEST_IDX]}"\n')

for ref in BEST_REFS:
    r = next((x for x in results if x['ref_id']==ref['id'] and x['test_idx']==TEST_IDX), None)
    if r is None: continue
    print(f'\n>>> Референс {ref["id"]} (rating={ref["rating"]}/5, RTF={r["rtf"]:.2f})')
    display(Audio(r['path']))

In [ ]:
# Сравнение на анекдоте Штирлиц (TEST_IDX=1)
TEST_IDX = 1
print(f'\n>>> Тестовая фраза: "{TEST_TEXTS[TEST_IDX]}"\n')

for ref in BEST_REFS:
    r = next((x for x in results if x['ref_id']==ref['id'] and x['test_idx']==TEST_IDX), None)
    if r is None: continue
    print(f'\n>>> Референс {ref["id"]} (rating={ref["rating"]}/5, RTF={r["rtf"]:.2f})')
    display(Audio(r['path']))

## 7. Выбрать лучший референс

После прослушивания — впиши какой референс даёт лучший клон Бурунова.

In [ ]:
# ⚠️ ПОМЕНЯЙ ПОСЛЕ ПРОСЛУШИВАНИЯ
# Варианты: 'ref_0043', 'ref_0293', 'ref_0054', 'ref_0025', 'ref_0292'
BEST_REF_ID = 'ref_0043'  # rating 5 — должен быть лучшим по умолчанию

best_ref = next(r for r in BEST_REFS if r['id']==BEST_REF_ID)
BEST_REF_AUDIO = os.path.join(REFS_DIR, best_ref['wav'])
BEST_REF_TEXT = best_ref['text']

print(f'✅ Лучший референс: {BEST_REF_ID}')
print(f'   Файл: {BEST_REF_AUDIO}')
print(f'   Текст: "{BEST_REF_TEXT}"')
print(f'   Rating: {best_ref["rating"]}/5')

## 8. Сводка RTF (real-time factor)

In [ ]:
# RTF только для лучшего референса
best_results = [r for r in results if r['ref_id']==BEST_REF_ID]

print('=== RTF для лучшего референса ===')
print(f'{"Фраза":<55} {"Синтез":<8} {"Аудио":<8} {"RTF":<6}')
print('-' * 80)
for r in best_results:
    print(f'{r["test_text"][:53]:<55} {r["synth_time"]:<8.1f} {r["audio_dur"]:<8.1f} {r["rtf"]:<6.2f}')

avg_rtf = sum(r['rtf'] for r in best_results) / len(best_results)
print(f'\nСредний RTF: {avg_rtf:.2f}')
if avg_rtf < 1.0:
    print('✅ real-time — F5-TTS можно юзать live на G1')
elif avg_rtf < 2.0:
    print('🟡 небольшая задержка — терпимо для демо')
else:
    print('🔴 медленно — лучше генерить wav заранее, не live')

## 9. Сгенерировать пресеты анекдотов + реплик кофе

Используем лучший референс. 16 файлов: 5 анекдотов × (intro+joke) + 6 реплик для кофе.

In [ ]:
PRESETS = {
    'shtirlits_intro': 'Ну, значицца, Штирлиц... он вообще почти весь восемьдесят шестой по телевизору был, ну вы помните.',
    'shtirlits_joke': 'Штирлиц подошёл к окну. Из окна дуло. Штирлиц закрыл окно. Дуло исчезло.',
    'volodka_intro': 'А вот Вовочка... ну, который из школы... этот, как его... хулиган.',
    'volodka_joke': 'Учительница спрашивает Вовочку: какую оценку тебе поставить? Пять, Марьиванна! Ну, ты ещё посчитай. Ладно, четыре, но не меньше!',
    'rzhevsky_intro': 'Ну этот... поручик наш... Ржевский, да... он вообще к дамам постоянно лез.',
    'rzhevsky_joke': 'Поручик Ржевский на балу подходит к Наташе Ростовой: Наташа, а Наташа, давайте я вас поцелую! Поручик, я же дама! Ну, тем более!',
    'new_russians_intro': 'А это ещё тогда, в восемьдесят шестом, кооперативы пошли, ну, эти... в малиновых пиджаках.',
    'new_russians_joke': 'Встречаются два новых русских. Один говорит: Я вчера Запорожец купил! Ну и как? Крылья бабочки, малиновый цвет, всё дела!',
    'chapaev_intro': 'Ну, Чапаев, понятное дело... он вообще к нам из гражданской войны пришёл, но мы его любим.',
    'chapaev_joke': 'Чапаев спрашивает Петьку: Петька, рубль есть? Нет, Василий Иваныч. А два есть? Тоже нет. Ну вот, а ты говоришь — капиталистическая революция.',
    'coffee_intro': 'Угу, щас, Олег Тарасыч... кофеварку найду.',
    'coffee_obstacle': 'Бля, тут кто-то стоит... дай пройти.',
    'coffee_no_cup': 'Не вижу никакой чашки, Олег. Где кофе-то?',
    'coffee_got_it': 'Взял. Сейчас принесу.',
    'coffee_dropped': 'Ой... выронил, бля.',
    'coffee_done': 'Вот ваш кофе, Олег. Не обожгись, бля.',
}

os.makedirs('/content/presets', exist_ok=True)
preset_results = []

print('Генерация пресетов с лучшим референсом...')
for name, text in PRESETS.items():
    out = f'/content/presets/{name}.wav'
    print(f'  → {name}')
    try:
        t0 = time.time()
        wav, sr, _ = tts.infer(
            ref_file=BEST_REF_AUDIO,
            ref_text=BEST_REF_TEXT,
            gen_text=text,
            file_wave=out,
        )
        dt = time.time() - t0
        audio, sr_out = sf.read(out)
        dur = len(audio) / sr_out
        print(f'    ✅ {dur:.1f}сек, синтез {dt:.1f}с')
        preset_results.append({'name': name, 'duration': dur, 'synth_time': dt, 'path': out, 'text': text})
    except Exception as e:
        print(f'    ❌ {e}')

print(f'\n✅ {len(preset_results)}/{len(PRESETS)} пресетов сгенерировано')

## 10. Прослушать пресеты

In [ ]:
for r in preset_results:
    print(f'\n>>> {r["name"]} ({r["duration"]:.1f}сек)')
    print(f'    {r["text"]}')
    display(Audio(r['path']))

## 11. Сохранить manifest + архив для G1

In [ ]:
import json, shutil

manifest_out = {
    'reference_audio': f'{best_ref["wav"]}',
    'reference_text': BEST_REF_TEXT,
    'reference_rating': best_ref['rating'],
    'reference_duration': best_ref['duration'],
    'dataset': 'burunov-voice/burunov-only-v2',
    'transcription': 'manual (друг кореша, 2026-07-08)',
    'model': 'hotstone228/F5-TTS-Russian',
    'device': 'colab-T4',
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'avg_rtf': avg_rtf,
    'presets': [
        {'name': r['name'], 'file': f"{r['name']}.wav", 'duration_s': r['duration'], 'text': r['text']}
        for r in preset_results
    ],
}
with open('/content/presets/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_out, f, ensure_ascii=False, indent=2)
print('✅ manifest.json сохранён')

# Архив для G1
shutil.copy(BEST_REF_AUDIO, f'/content/presets/{best_ref["wav"]}')
with open(f'/content/presets/reference_text.txt', 'w', encoding='utf-8') as f:
    f.write(BEST_REF_TEXT)
!cd /content && tar -czf burunov_presets.tar.gz presets/
!ls -lh /content/burunov_presets.tar.gz

print('\n📥 Скачай burunov_presets.tar.gz через панель файлов слева')
print('   Это всё что нужно для деплоя на G1')

## 12. Что дальше на G1

```bash
# С ноутбука на G1:
scp burunov_presets.tar.gz unitree@G1_IP:~/burunov-joke-bot/

# На G1:
cd ~/burunov-joke-bot
mkdir -p data/preset_wav
tar -xzf burunov_presets.tar.gz -C data/preset_wav --strip-components=1

# Если RTF < 1 — можно live через F5-TTS:
# pip install f5-tts && python3 f5_tts_server.py

# Если RTF > 1 — играем готовые wav через preset_player (скоро будет)
#```